# ЛР №1. Часть 2. Диалог двух LLM-агентов с разными системными промптами

🔒 Ячейки с замком — НЕ менять, только запускать (Shift+Enter).

✏️ В ячейках с карандашом — править только выделенные строки.

In [12]:
# установка библиотек
# 🔒 НЕ МЕНЯТЬ — просто выполнить (один раз)
%pip install -q -U openai python-dotenv 
#Это магическая команда Jupyter — служебная инструкция для ноутбука. Все такие команды начинаются с символа % (или !). 
                                        #здесь %pip - запуск пакетного менеджера pip внутри текущего ядра ноутбука. 
                                        #Префикс % гарантирует установку именно в тот Python, который выполняет ноутбук
                                        #-q («quiet») — тихий режим: скрывает служебный вывод установки, чтобы не засорять ноутбук
                                        # -U («upgrade») — если пакет уже установлен, обновить его до последней версии
                                        #  openai - первый пакет: клиент для OpenAI-совместимых API. Именно через него мы подключаемся к Yandex Cloud (client.responses.create(...))
                                        # python-dotenv - второй пакет: загрузка переменных из файла .env (функция load_dotenv()) — чтобы не хранить API-ключ в коде

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# создание файла .env, куда будет размещен api-ключ
#🔒 НЕ МЕНЯТЬ — выполнить ОДИН РАЗ, затем открыть файл .env слева
# в проводнике Jupyter и вписать туда свои FOLDER_ID и API_KEY
with open(".env", "w", encoding="utf-8") as f:  # with - контекстный менеджер: открывает файл и гарантирует его закрытие после выполнения блока — даже если внутри произойдёт ошибка. 
                                                #Явно писать f.close() не нужно
                                                # .env - Имя файла. Путь относительный — файл создаётся в текущей рабочей папке (в Jupyter — там же, где лежит ноутбук). Имя с точкой — соглашение для скрытых файлов-настроек
                                                # w - Режим записи (write): если файла нет — создаст; если есть — полностью перезапишет (сотрёт содержимое!)
                                                # encoding="utf-8" - Кодировка: русский текст будет записан и прочитан корректно на любой ОС. На Windows без этого параметра файл может записаться в cp1251, и другие программы прочтут его с «кракозябрами»
                                                # Связывает открытый файл с переменной f — через неё внутри блока вызывается f.write(...)
    f.write("YC_FOLDER_ID=ВСТАВЬТЕ_ID_ПАПКИ\n")
    f.write("YC_API_KEY=ВСТАВЬТЕ_API_КЛЮЧ\n")
print("Файл .env создан. Откройте его и впишите свои данные!")

⚠️ Ключ нельзя писать прямо в код

In [13]:
# 🔒 НЕ МЕНЯТЬ, здесь проходит  настройка клиента
import os  # работа с переменными окружения и файлами
from dotenv import load_dotenv  # чтение ключей из файла .env
import openai  # клиент формата OpenAI (подходит и для Яндекса)

load_dotenv(override=True) # загружаем значения из .env в окружение

FOLDER_ID = os.getenv("YC_FOLDER_ID")  # идентификатор папки в Yandex Cloud
API_KEY   = os.getenv("YC_API_KEY")  # API-ключ сервисного аккаунта

# Проверяем, что ключи заполнены и это не шаблонные заглушки
# assert-Ключевое слово Python: проверяет условие.  
#Если условие истинно (True) — программа идёт дальше. Если ложно (False) — выбрасывает исключение AssertionError и останавливается
# , \ - Запятая разделяет условие и сообщение об ошибке. Обратный слэш \ означает «строка кода продолжается на следующей строке» — так длинную команду разбивают на две для читаемости
assert FOLDER_ID and API_KEY and "ВСТАВЬТЕ" not in (FOLDER_ID + API_KEY), \
    "Заполните файл .env реальными значениями FOLDER_ID и API_KEY!"  

# Создаём клиент: Yandex Cloud поддерживает OpenAI-совместимый API,                                                               
# поэтому используется стандартная библиотека openai, меняется только адрес
client = openai.OpenAI(
	api_key=API_KEY,   # ключ уходит в заголовок авторизации
	base_url="https://ai.api.cloud.yandex.net/v1", # адрес API Yandex Cloud 
)

# Идентификатор модели в формате Яндекса: gpt://<папка>/<модель>/<версия>
# yandexgpt-lite — быстрее и дешевле; можно заменить на yandexgpt или другую доступную модель
MODEL_URI = f"gpt://{FOLDER_ID}/yandexgpt-lite/latest"
# Проверка связи: одноразовый запрос к модели
resp = client.responses.create(model=MODEL_URI, # выбранная модель
                               input="Подтверди одной фразой, что подключение работает.",  # текст запроса
                              )
print("✅", resp.output_text) # выводим ответ модели

AuthenticationError: Error code: 401 - {'error': "UNAUTHENTICATED: Unknown api key 'AQVN****x2Zm (3BB54087)'"}

In [14]:
print("FOLDER_ID:", FOLDER_ID)
print("API_KEY задан:", bool(API_KEY))
print("Длина ключа:", len(API_KEY) if API_KEY else 0)
print("MODEL_URI:", MODEL_URI)
print("client определён:", "client" in globals())

FOLDER_ID: b1ga9e9r2juv9aktc4ou
API_KEY задан: True
Длина ключа: 40
MODEL_URI: gpt://b1ga9e9r2juv9aktc4ou/yandexgpt-lite/latest
client определён: True


In [ ]:
# класс Agent
# 🔒 НЕ МЕНЯТЬ — это и есть «обёртка над Responses API» из цели работы
# ---------------------------------------------------------------------
# Ключевое слово `class` создаёт новый ТИП данных (как int или str),
# но со своим поведением. Имя класса пишется с Заглавной буквы —
# это соглашение Python (PEP 8), чтобы отличать классы от переменных.
#Строка в тройных кавычках сразу после class — это docstring. 
#Это описание класса, которое показывается при наведении мыши или вызове help(Agent). Не влияет на работу кода, но обязательно для читаемости.

class Agent:
    """Агент — обёртка над Responses API с памятью через previous_response_id."""
    # ---------------------------------------------------------
    # __init__ — это КОНСТРУКТОР. Специальный метод (начинается
    # и заканчивается двойным подчёркиванием = «dunder method»).
    # Он вызывается АВТОМАТИЧЕСКИ каждый раз, когда вы пишете:
    #     agent = Agent(name="Оптимист", instructions="...")
    # Его задача — настроить начальные значения атрибутов объекта.
    # ---------------------------------------------------------
    def __init__(self, name: str, instructions: str):
        # `def` — ключевое слово для определения метода (функции внутри класса)
        # `self` — ОБЯЗАТЕЛЬНЫЙ первый параметр любого метода класса.
        #         Это ссылка на сам объект, который сейчас создаётся.
        #         Через self мы сохраняем данные ВНУТРЬ конкретного объекта.
        #         При вызове Agent(...) Python передаёт self автоматически,
        #         вы его НЕ указываете в скобках.
        # `name: str` — аннотация типа. Говорит «ожидается строка».
        #               Python не проверяет это принудительно, но IDE
        #               подсказывает ошибки, и улучшается читаемость.
        # `instructions: str` — второй параметр, тоже строка.

        # Сохраняем параметры как АТРИБУТЫ объекта.
        # self.name — это поле, привязанное к КОНКРЕТНОМУ агенту.
        # У каждого агента будет СВОЁ значение self.name.
        self.name = name                    # имя для стенограммы
        self.instructions = instructions    # системный промпт = «личность»
        self.last_response_id = None        # id последнего ответа = память диалога
        # `None` — специальное значение Python = «ничего / пусто».
        # В начале диалога памяти нет, поэтому ставим None.
        # Потом сюда запишется реальный id ответа модели.

    # ---------------------------------------------------------
    # Обычный метод класса. Принимает self + свои параметры.
    # Вызывается так:   agent.reply("привет")
    # Python автоматически подставит agent как self.
    # ---------------------------------------------------------
    def reply(self, message: str) -> str:
        # `-> str` — аннотация возвращаемого типа.
        # Метод обещает вернуть строку.

        response = client.responses.create(
            model=MODEL_URI,                                # какая модель отвечает
            instructions=self.instructions,                 # берём из АТРИБУТА объекта
            input=[{"role": "user", "content": message}],   # новое сообщение
            previous_response_id=self.last_response_id,     # подхватываем память
            # Если last_response_id == None (первый вызов),
            # API просто игнорирует этот параметр — диалог начинается с нуля.
        )

        # Обновляем память: запоминаем id ТЕКУЩЕГО ответа,
        # чтобы следующий вызов reply() продолжил эту же нить.
        self.last_response_id = response.id

        return response.output_text  # возвращаем текст ответа модели

    # ---------------------------------------------------------
    # Ещё один метод. Сбрасывает память агента.
    # Вызывается так:   agent.reset()
    # Обратите внимание: у reset НЕТ параметров кроме self.
    # ---------------------------------------------------------
    def reset(self):
        """Сбрасывает нить диалога (разговор с чистого листа)."""
        self.last_response_id = None
        # Присваиваем None → при следующем reply() previous_response_id
        # снова будет None → API начнёт новую нить.

In [ ]:
# создаем двух агентов
# ✏️ ПРАВИТЬ ТОЛЬКО ТЕКСТЫ В КАВЫЧКАХ
optimist = Agent(
    name="Оптимист",  # имя для стенограммы
    instructions="Ты — Оптимист. Находи плюсы в любой ситуации, поддерживай новые идеи. Отвечай кратко: 2–3 предложения.", # системный промпт №1
)

skeptic = Agent(
    name="Скептик",  # имя для стенограммы
    instructions="Ты — Скептик. Анализируй риски, задавай уточняющие вопросы, требуй аргументов. Отвечай кратко: 2–3 предложения.", # системный промпт №2
)
print("Агенты созданы:", optimist.name, "и", skeptic.name)

In [ ]:
# функция диалога
# 🔒 НЕ МЕНЯТЬ — каждый агент получает реплику другого как входное сообщение
def run_dialogue(agent_a: Agent, agent_b: Agent, topic: str, rounds: int = 3):
    """
    Организует диалог: реплика одного агента становится входом для другого.
    Возвращает стенограмму — список кортежей (имя, реплика).
    """
    transcript = [] # стенограмма для отчёта
    message_for_a = f"Тема дискуссии: «{topic}». Выскажи своё мнение первым."  # стартовое сообщение для агента A
    for _ in range(rounds): # rounds = число пар реплик A→B
        reply_a = agent_a.reply(message_for_a) # 1) отвечает агент A
        transcript.append((agent_a.name, reply_a)) #    сохраняем в стенограмму
        print(f"🗣 {agent_a.name}: {reply_a}\n") #    выводим на экран

        reply_b = agent_b.reply(f"{agent_a.name} говорит: «{reply_a}». Выскажи своё мнение.") # 2) агент B получает реплику A
        transcript.append((agent_b.name, reply_b)) #    сохраняем в стенограмму
        print(f"🗣 {agent_b.name}: {reply_b}\n") #    выводим на экран

        message_for_a = f"{agent_b.name} отвечает: «{reply_b}». Продолжи дискуссию."  # 3) для следующего хода агента A
    return transcript # возвращаем полную стенограмму

In [ ]:
# запускаем дискуссию
# ✏️ ПРАВИТЬ ТОЛЬКО ТЕМУ И ЧИСЛО РАУНДОВ
TOPIC  = "Внедрение ИИ-агентов в учебный процесс университета" # тема спора
ROUNDS = 3  # пар реплик

transcript = run_dialogue(optimist, skeptic, TOPIC, rounds=ROUNDS) # запуск!

In [ ]:
# эксперимент с памятью
# 🔒 НЕ МЕНЯТЬ — демонстрация механизма previous_response_id
print("С памятью:") # нить диалога ещё жива
print("→", skeptic.reply("Напомни в одном предложении самую первую реплику собеседника."))  # спрашиваем о начале разговор

skeptic.reset()  # сбрасываем контекст
print("\nПосле сброса контекста (тот же вопрос):") # нить удалена
print("→", skeptic.reply("Напомни в одном предложении самую первую реплику собеседника.")) # модель «забыла» прошлое

In [ ]:
# эксперимент «сила системного промпта»
# ✏️ МОЖНО МЕНЯТЬ РОЛИ И ВОПРОС
QUESTION = "Что такое нейросеть?"  # один и тот же вопрос для всех ролей
ROLES = { # три разных системных промпта
    "Журналист":  "Ты журналист: излагай факты нейтрально.",
    "Маркетолог": "Ты маркетолог: продавай идею ярко и эмоционально.",
    "Юрист":      "Ты юрист: указывай риски и ограничения.",
}
for role, prompt in ROLES.items(): # проходим по всем ролям по очереди
    print(f"--- {role} ---") # заголовок для наглядности
    print(Agent(role, prompt).reply(QUESTION), "\n") # одна модель — разные ответы

In [11]:
# индивидуальное задание
# ✏️ СОЗДАЙТЕ СВОЮ ПАРУ АГЕНТОВ (примеры: рекрутёр/кандидат, продавец/покупатель, тьютор/студент)
agent_1 = Agent(name="...", instructions="...")
agent_2 = Agent(name="...", instructions="...")

transcript_my = run_dialogue(agent_1, agent_2, topic="...", rounds=3)

🗣 ...: Вы подняли несколько важных вопросов, связанных с влиянием цифровизации на образование.

**1. Создание образовательной среды, где онлайн-технологии дополняют традиционные методы обучения.**

Баланс между цифровыми и очными форматами обучения действительно важен. Онлайн-технологии могут обогатить учебный процесс, предоставляя доступ к дополнительным ресурсам и интерактивным материалам. Однако они не могут полностью заменить живое общение и личное взаимодействие между учащимися и преподавателями.

Стратегии для сохранения баланса:
* **Интеграция:** использование онлайн-ресурсов как дополнения к традиционным урокам, а не замена им. Например, видеолекции или онлайн-консультации могут обогатить учебный процесс, не заменяя личные встречи с преподавателем.
* **Гибкость:** предоставление учащимся выбора формата обучения. Некоторые предметы могут лучше усваиваться в традиционной форме, в то время как другие могут выиграть от онлайн-формата.
* **Смешанное обучение:** сочетание очных и онл

In [ ]:
# сохранение стенограммы для отчёта
# 🔒 НЕ МЕНЯТЬ
import json # сохранение стенограммы диалога в JSON
all_transcripts = transcript + transcript_my            # объединяем обе стенограммы
with open("transcript.json", "w", encoding="utf-8") as f:  # открываем файл на запись
    json.dump(all_transcripts, f, ensure_ascii=False, indent=2)  # пишем в JSON
print("💾 Стенограмма сохранена в transcript.json")      # подтверждаем сохранение

## Если что-то не работает:
- 401/403 — неверный ключ или нет роли `ai.editor` у сервисного аккаунта;
- при ошибке 404 возьмите актуальный пример кода из интерфейса AI Studio (кнопка "Код")
- 429 — слишком частые запросы: уменьшите ROUNDS до 2;
- «ВСТАВЬТЕ_ID_ПАПКИ» в ошибке — заполните .env и перезапустите ядро.